## **Term Project #2 (200 points)**
- Instructor: [Jaeung Sim](https://jaeungs.github.io/) (University of Connecticut)
- Course: OPIM 5671 Data Mining and Time Series Forecasting
- Release Date: April 8 (Wed), 2026
- Submission Due: April 21 (Tue), 2026
- Submission Materials: Python notebook, presentation slides

**Objective**
* Explore real-world text data and draw novel insights using natural language processing.
* The performance of this project will be evaluated based on the communication about the significance of your problem and insights, as well as the implementation of text mining techniques.


#### **Context: Australian Election 2019 Tweets**

**Introduction to the Dataset**
* **Source:** Australian Election 2019 Tweets at Kaggle (<https://www.kaggle.com/datasets/taniaj/australian-election-2019-tweets>)
* **About this file**
  * Over 180,000 tweets collected using Twitter API keyword search between May 10, 2019, and May 20, 2019.
  * Column Description
    1. `created_at`: Date and time of tweet creation
    1. `id`: Unique ID of the tweet
    1. `full_text`: Full tweet text
    1. `retweet_count`: Number of retweets
    1. `favorite_count`: Number of likes
    1. `user_id`: User ID of tweet creator
    1. `user_name`: Username of tweet creator
    1. `user_screen_name`: Screen name of tweet creator
    1. `user_description`: Description on tweet creator's profile
    1. `user_location`: Location given on tweet creator's profile
    1. `user_created_at`: Date the tweet creator joined Twitter

#### **Provide your team information**

* **Team members:** [Fill in the blank]
* **Each member's contribution:**
  * [Teammate 1]: [Fill in the blank]
  * [Teammate 2]: [Fill in the blank]
  * [Teammate 3]: [Fill in the blank]
  * [Teammate 4]: [Fill in the blank]

---
## **Section 0: Setup & Install Dependencies**

In [1]:
# Install required libraries
!pip install kagglehub textblob nrclex wordcloud matplotlib seaborn scikit-learn gensim pyLDAvis contractions --quiet
!pip install nltk --quiet

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print('All dependencies installed successfully.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.4 MB/s eta 0:00:00
All dependencies installed successfully.


---
## **Section 1: Data Processing**

In [2]:
# --- Import core libraries ---
import os
import re
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
warnings.filterwarnings('ignore')

# NLTK imports
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

print('Core libraries imported.')

Core libraries imported.


In [3]:
# --- Download Dataset from Kaggle ---
import kagglehub

path = kagglehub.dataset_download('taniaj/australian-election-2019-tweets')
print('Path to dataset files:', path)

files = os.listdir(path)
print('Files available:', files)

Using Colab cache for faster access to the 'australian-election-2019-tweets' dataset.
Path to dataset files: /kaggle/input/australian-election-2019-tweets
Files available: ['location_geocode.csv', 'auspol2019.csv']


In [4]:
# --- Load the primary dataset ---
# The dataset contains two CSV files; we focus on the main one with full_text
csv_files = [f for f in files if f.endswith('.csv')]
print('CSV files found:', csv_files)

# Load the larger CSV (primary tweet data), which is 'auspol2019.csv'
primary_csv_file = 'auspol2019.csv'
if primary_csv_file in csv_files:
    csv_path = os.path.join(path, primary_csv_file)
    df = pd.read_csv(csv_path, encoding='ISO-8859-1')
    print(f'DataFrame dimensions: {df.shape}')
    print('\nColumn names:')
    print(df.columns.tolist())
    print('\nFirst 3 rows:')
    print(df.head(3))
else:
    print(f"Error: '{primary_csv_file}' not found in the dataset folder.")


CSV files found: ['location_geocode.csv', 'auspol2019.csv']
DataFrame dimensions: (183379, 11)

Column names:
['created_at', 'id', 'full_text', 'retweet_count', 'favorite_count', 'user_id', 'user_name', 'user_screen_name', 'user_description', 'user_location', 'user_created_at']

First 3 rows:
            created_at                   id  \
0  2019-05-20 09:13:44  1130401208756187136   
1  2019-05-20 09:13:43  1130401205367140357   
2  2019-05-20 09:13:33  1130401162782371841   

                                           full_text  retweet_count  \
0  After the climate election: shellshocked green...            0.0   
1  @narendramodi @smritiirani Coverage of indian ...            0.0   
2  @workmanalice Do you know if Facebook is relea...            0.0   

   favorite_count      user_id        user_name user_screen_name  \
0             0.0   92484856.0     PIPELINEPETE         jocksjig   
1             0.0  775647396.0  Narinder Parmar      nparmar1957   
2             0.0      56873

In [5]:
# --- Check for null values ---
print('Null value proportions per column:')
print(df.isnull().mean().round(4))

print(f'\nTotal rows before handling nulls: {len(df)}')

Null value proportions per column:
created_at          0.0000
id                  0.0000
full_text           0.0000
retweet_count       0.0000
favorite_count      0.0000
user_id             0.0000
user_name           0.0001
user_screen_name    0.0000
user_description    0.0857
user_location       0.2012
user_created_at     0.0001
dtype: float64

Total rows before handling nulls: 183379


In [6]:
# --- Handle missing values ---
# Since our analysis focuses on full_text (no null values),
# we keep all rows but fill missing values to preserve data completeness.
# This approach avoids losing top tweeters who may have missing metadata.

df['retweet_count'] = df['retweet_count'].fillna(0)
df['favorite_count'] = df['favorite_count'].fillna(0)
df['user_description'] = df['user_description'].fillna('Unknown')
df['user_location'] = df['user_location'].fillna('Unknown')
df['user_name'] = df['user_name'].fillna('Unknown')
df['user_screen_name'] = df['user_screen_name'].fillna('Unknown')

# Convert date columns
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['user_created_at'] = pd.to_datetime(df['user_created_at'], errors='coerce')

# Remove rows where full_text is missing (our primary column)
df = df.dropna(subset=['full_text'])

print(f'Rows after cleaning: {len(df)}')
print('Remaining null values:')
print(df.isnull().sum())

Rows after cleaning: 183379
Remaining null values:
created_at           9
id                   0
full_text            0
retweet_count        0
favorite_count       0
user_id              9
user_name            0
user_screen_name     0
user_description     0
user_location        0
user_created_at     18
dtype: int64


In [7]:
# --- Text Processing: Define process_text function ---
# Following the standard NLP pipeline from the course notebooks:
# 1. Lowercase  2. Remove special chars  3. Tokenize  4. Remove stopwords  5. Lemmatize

lemmatizer = WordNetLemmatizer()
STOPWORDS = set(stopwords.words('english'))

# Add domain-specific stopwords (election hashtags and noise)
CUSTOM_STOPWORDS = {
    'amp', 'abc', 'ausvotes', 'auspol', 'auspol2019', 'ausvotes2019', 'ausvotes19',
    'ausvote19', 'australiavotes', 'australiavotes2019', '7news',
    'election2019results', 'election', 'qldpol', 'australia', 'australian',
    'via', 'u', 'lnp', 'would', 'tony', 'clive', '2019',
    'auspol19', 'election2019', 'australiadecides', 'rt'
}
STOPWORDS.update(CUSTOM_STOPWORDS)

def process_text(text):
    """Clean and tokenize tweet text."""
    if not isinstance(text, str):
        return ''
    # Lowercase
    text = text.lower()
    # Tokenize
    tokens = word_tokenize(text)
    new_text = []
    for word in tokens:
        # Skip empty tokens (guard against IndexError on word[0])
        if not word:
            continue
        # Remove stopwords
        if word in STOPWORDS:
            continue
        # Filter special characters: @mentions, http URLs, emoji (\xf0 prefix)
        if word[0] == '@' or word[:4] == 'http' or word[0] == '\xf0':
            continue
        # Remove punctuation-only tokens
        if all(c in string.punctuation for c in word):
            continue
        # Keep only alphabetic tokens (removes numbers, mixed tokens)
        if not word.isalpha():
            continue
        # Lemmatize
        word = lemmatizer.lemmatize(word)
        if len(word) > 2:  # drop very short tokens
            new_text.append(word)
    return ' '.join(new_text)

# Apply text processing
print('Processing text... (this may take a minute)')
df['clean_text'] = df['full_text'].apply(process_text)

print('Text processing complete.')
print(f'\nSample clean_text values:')
df[['full_text', 'clean_text']].head(5)


Processing text... (this may take a minute)
Text processing complete.

Sample clean_text values:


,full_text,clean_text
0,After the climate election: shellshocked green...,climate shellshocked green group remain resolute
1,@narendramodi @smritiirani Coverage of indian ...,narendramodi smritiirani coverage indian chann...
2,@workmanalice Do you know if Facebook is relea...,workmanalice know facebook releasing looked mi...
3,@vanbadham We all understand we have a compuls...,vanbadham understand compulsory preference sys...
4,"Shares were mixed in Asia, with India and Aust...",share mixed asia india leading gain region fol...


In [8]:
# --- Verify the clean_text column ---
print('clean_text column stats:')
print(f'  Total rows: {len(df)}')
print(f'  Empty clean_text: {(df["clean_text"] == "").sum()}')
print(f'  Null clean_text: {df["clean_text"].isnull().sum()}')

# Remove rows where clean_text is empty after processing
df = df[df['clean_text'].str.strip() != '']
df = df.reset_index(drop=True)
print(f'\nFinal dataset size: {df.shape}')

clean_text column stats:
  Total rows: 183379
  Empty clean_text: 4091
  Null clean_text: 0

Final dataset size: (179288, 12)


---
## **Section 2: Feature Generation**

In [9]:
# --- 2.1 TextBlob Sentiment Analysis ---
from textblob import TextBlob

print('Computing TextBlob sentiment...')
df['TextBlob_Sentiment'] = df['full_text'].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity
)
df['TextBlob_Subjectivity'] = df['full_text'].apply(
    lambda x: TextBlob(str(x)).sentiment.subjectivity
)

print('TextBlob Sentiment Summary Statistics:')
print(df['TextBlob_Sentiment'].describe())

Computing TextBlob sentiment...
TextBlob Sentiment Summary Statistics:
count    179288.000000
mean          0.078480
std           0.284651
min          -1.000000
25%           0.000000
50%           0.000000
75%           0.200000
max           1.000000
Name: TextBlob_Sentiment, dtype: float64


In [ ]:
# --- 2.2 NRC Emotion Features ---
from nrclex import NRCLex

def get_emotions(text):
    """Extract NRC emotion scores from text using the updated NRCLex API."""
    try:
        emo = NRCLex()
        emo.load_raw_text(str(text))
        return emo.raw_emotion_scores
    except Exception:
        return {}

print('Computing NRC emotion features... (this may take a few minutes)')
df['NRC_Emotions'] = df['full_text'].astype(str).apply(get_emotions)

# Expand emotion dictionary into separate columns
emotion_df = df['NRC_Emotions'].apply(pd.Series).fillna(0)
df = pd.concat([df.drop(columns=[c for c in emotion_df.columns if c in df.columns], errors='ignore'), emotion_df], axis=1)

# Ensure all expected emotion columns exist
emotion_cols = ['fear', 'anger', 'anticipation', 'trust', 'surprise', 'sadness', 'joy', 'disgust', 'positive', 'negative']
for emotion in emotion_cols:
    if emotion not in df.columns:
        df[emotion] = 0

print('NRC emotion features created.')
df[['clean_text', 'TextBlob_Sentiment'] + emotion_cols].head(3)


Computing NRC emotion features... (this may take a few minutes)


In [ ]:
# --- Verify NRC Emotion Scores ---
print('\nSample NRC_Emotions for first 5 rows:')
for i, emotions in enumerate(df['NRC_Emotions'].head(5)):
    print(f'Row {i}: {emotions}')

# Count how many tweets have at least one non-zero emotion score
non_zero_emotions_count = df[emotion_cols].sum(axis=1).astype(bool).sum()
print(f'\nNumber of tweets with at least one non-zero emotion score: {non_zero_emotions_count} out of {len(df)}')

if non_zero_emotions_count == 0:
    print('This suggests that NRCLex is not detecting any emotions in the provided text, leading to zero average scores across all categories.')


In [ ]:
# --- 2.3 Text Complexity & Engagement Features ---
import math

# Word count and character count
df['word_count'] = df['clean_text'].apply(lambda x: len(str(x).split()))
df['char_count'] = df['full_text'].apply(lambda x: len(str(x)))

# Lexical diversity (unique words / total words)
df['lexical_diversity'] = df['clean_text'].apply(
    lambda x: len(set(str(x).split())) / len(str(x).split()) if len(str(x).split()) > 0 else 0
)

# Flesch Reading Ease score
def flesch_reading_ease(text):
    """Compute Flesch Reading Ease score."""
    words = str(text).split()
    if len(words) == 0:
        return 0
    # Count syllables (approximation)
    def syllable_count(word):
        word = word.lower()
        count = 0
        vowels = 'aeiouy'
        if word[0] in vowels:
            count += 1
        for i in range(1, len(word)):
            if word[i] in vowels and word[i-1] not in vowels:
                count += 1
        if word.endswith('e'):
            count -= 1
        if count == 0:
            count = 1
        return count
    sentences = max(1, text.count('.') + text.count('!') + text.count('?'))
    total_syllables = sum(syllable_count(w) for w in words)
    score = 206.835 - 1.015*(len(words)/sentences) - 84.6*(total_syllables/len(words))
    return round(score, 2)

df['flesch_reading_ease'] = df['full_text'].apply(flesch_reading_ease)

# Account age (days from user_created_at to tweet created_at)
df['account_age_days'] = (df['created_at'] - df['user_created_at']).dt.days

# Engagement features
df['popularity_score'] = df['retweet_count'] + df['favorite_count']
df['has_hashtag'] = df['full_text'].apply(lambda x: 1 if '#' in str(x) else 0)
df['has_mention'] = df['full_text'].apply(lambda x: 1 if '@' in str(x) else 0)
df['has_url'] = df['full_text'].apply(lambda x: 1 if 'http' in str(x) else 0)

# Hashtag count
df['hashtag_count'] = df['full_text'].apply(lambda x: len(re.findall(r'#\w+', str(x))))

# Mention count
df['mention_count'] = df['full_text'].apply(lambda x: len(re.findall(r'@\w+', str(x))))

# Sentiment category (positive/neutral/negative)
df['sentiment_category'] = df['TextBlob_Sentiment'].apply(
    lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral')
)

print('All features generated. Summary of new columns:')
feature_cols = ['word_count', 'char_count', 'lexical_diversity', 'flesch_reading_ease',
                'account_age_days', 'popularity_score', 'has_hashtag', 'has_mention',
                'hashtag_count', 'mention_count', 'sentiment_category']
df[feature_cols].describe(include='all')

---
## **Section 3: Analysis to Derive Insights**

### **3.1 Exploratory Data Analysis (EDA)**

In [ ]:
# --- Word Frequency Bar Chart ---
all_words = ' '.join(df['clean_text'].dropna()).split()
word_freq = Counter(all_words)
top_words = word_freq.most_common(25)
words, counts = zip(*top_words)

# Color gradient
colors = plt.cm.Blues_r(np.linspace(0.3, 0.9, len(words)))

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(words, counts, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Top 25 Most Frequent Words in Tweets', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Words', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.tick_params(axis='x', rotation=45)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('word_frequency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 5 words:', list(words[:5]))

In [ ]:
# --- Word Cloud ---
from wordcloud import WordCloud

text_combined = ' '.join(df['clean_text'].dropna())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Word cloud of clean_text
wc1 = WordCloud(
    background_color='white',
    stopwords=STOPWORDS,
    max_words=100,
    max_font_size=80,
    colormap='Blues',
    random_state=42,
    width=800, height=400
).generate(text_combined)
axes[0].imshow(wc1, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Word Cloud: Tweet Content (clean_text)', fontsize=13, fontweight='bold')

# Word cloud of user descriptions
desc_text = ' '.join(df['user_description'].dropna().astype(str).tolist())
wc2 = WordCloud(
    background_color='black',
    stopwords=STOPWORDS,
    max_words=80,
    max_font_size=60,
    colormap='Greens',
    random_state=42,
    width=800, height=400
).generate(desc_text)
axes[1].imshow(wc2, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Word Cloud: User Descriptions', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Tweet Activity Over Time ---
df_time = df.dropna(subset=['created_at']).copy()
df_time['date'] = df_time['created_at'].dt.date
df_time['hour'] = df_time['created_at'].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# By day
daily = df_time.groupby('date').size().reset_index(name='tweet_count')
daily_idx = range(len(daily))
axes[0].plot(daily_idx, daily['tweet_count'], marker='o', color='#1a78c2', linewidth=2, markersize=6)
axes[0].fill_between(daily_idx, daily['tweet_count'], alpha=0.2, color='#1a78c2')
axes[0].set_xticks(list(daily_idx))
axes[0].set_xticklabels([str(d) for d in daily['date']], rotation=45, ha='right')
axes[0].set_title('Daily Tweet Volume', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Number of Tweets')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# By hour
hourly = df_time.groupby('hour').size().reset_index(name='tweet_count')
colors_h = plt.cm.plasma(np.linspace(0.2, 0.9, 24))
axes[1].bar(hourly['hour'], hourly['tweet_count'], color=colors_h, edgecolor='white')
axes[1].set_title('Tweet Activity by Hour of Day', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Hour of Day (UTC)')
axes[1].set_ylabel('Number of Tweets')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('tweet_activity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Peak day: {daily.loc[daily["tweet_count"].idxmax(), "date"]} ({daily["tweet_count"].max()} tweets)')
print(f'Peak hour: {hourly.loc[hourly["tweet_count"].idxmax(), "hour"]}:00 ({hourly["tweet_count"].max()} tweets)')


In [ ]:
# --- Top Hashtags and Mentions ---
hashtags = re.findall(r'#(\w+)', ' '.join(df['full_text'].astype(str).tolist()))
mentions = re.findall(r'@(\w+)', ' '.join(df['full_text'].astype(str).tolist()))

top_hashtags = Counter([h.lower() for h in hashtags]).most_common(10)
top_mentions = Counter([m.lower() for m in mentions]).most_common(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Hashtags
h_tags, h_counts = zip(*top_hashtags)
axes[0].barh(h_tags[::-1], h_counts[::-1], color='#2196F3', edgecolor='white')
axes[0].set_title('Top 10 Hashtags', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Mentions
m_tags, m_counts = zip(*top_mentions)
axes[1].barh(m_tags[::-1], m_counts[::-1], color='#FF5722', edgecolor='white')
axes[1].set_title('Top 10 @Mentions', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('hashtags_mentions.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 hashtags:', dict(top_hashtags))
print('Top 10 mentions:', dict(top_mentions))

In [ ]:
# --- Engagement Analysis: Retweet and Favorite Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Retweet distribution (log scale)
rt_data = df['retweet_count'][df['retweet_count'] > 0]
axes[0].hist(rt_data, bins=50, color='#1976D2', edgecolor='white', alpha=0.85)
axes[0].set_title('Retweet Count Distribution\n(Non-zero only)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Retweet Count')
axes[0].set_ylabel('Frequency')
axes[0].set_yscale('log')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Favorite distribution
fav_data = df['favorite_count'][df['favorite_count'] > 0]
axes[1].hist(fav_data, bins=50, color='#FF9800', edgecolor='white', alpha=0.85)
axes[1].set_title('Favorite Count Distribution\n(Non-zero only)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Favorite Count')
axes[1].set_ylabel('Frequency')
axes[1].set_yscale('log')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('engagement_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Max retweets: {df["retweet_count"].max()}')
print(f'Max favorites: {df["favorite_count"].max()}')
print(f'Most retweeted tweet:\n{df.loc[df["retweet_count"].idxmax(), "full_text"]}')

In [ ]:
# --- Top 10 Most Influential Users ---
# Influence score = total retweets + favorites received
df['influence_score'] = df['retweet_count'] + df['favorite_count']
top_users = df.groupby('user_screen_name')['influence_score'].sum().nlargest(10).reset_index()
top_users.columns = ['user_screen_name', 'influence_score']

print('Top 10 Most Influential Users:')
print(top_users.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors_u = plt.cm.coolwarm(np.linspace(0, 1, len(top_users)))
ax.barh(top_users['user_screen_name'], top_users['influence_score'], color=colors_u)
ax.set_title('Top 10 Most Influential Users', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Influence Score (Retweets + Favorites)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('influential_users.png', dpi=150, bbox_inches='tight')
plt.show()

### **3.2 Sentiment Analysis**

In [ ]:
# --- TextBlob Sentiment Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['TextBlob_Sentiment'], bins=40, color='#26C6DA', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Neutral (0)')
axes[0].set_title('Distribution of TextBlob Sentiment Scores', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sentiment Score (Polarity)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Pie chart of sentiment categories
cat_counts = df['sentiment_category'].value_counts()
colors_pie = ['#4CAF50', '#9E9E9E', '#F44336']
axes[1].pie(cat_counts, labels=cat_counts.index, autopct='%1.1f%%',
            colors=colors_pie, startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Sentiment Category Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(df['TextBlob_Sentiment'].describe())

In [ ]:
# --- Sentiment Over Time ---
df_time2 = df.dropna(subset=['created_at']).copy()
df_time2['date'] = df_time2['created_at'].dt.date
daily_sentiment = df_time2.groupby('date')['TextBlob_Sentiment'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ds_idx = range(len(daily_sentiment))
ax.plot(ds_idx, daily_sentiment['TextBlob_Sentiment'],
        marker='o', linewidth=2.5, color='#F59E0B', markersize=8)
ax.fill_between(ds_idx, daily_sentiment['TextBlob_Sentiment'],
                daily_sentiment['TextBlob_Sentiment'].mean(), alpha=0.2, color='#F59E0B')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Average Daily Tweet Sentiment (May 10-20, 2019)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Average Sentiment Polarity')
ax.set_xticks(list(ds_idx))
ax.set_xticklabels([str(d) for d in daily_sentiment['date']], rotation=45, ha='right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Annotate election day
import datetime
election_day = datetime.date(2019, 5, 18)
if election_day in daily_sentiment['date'].values:
    ed_idx = list(daily_sentiment['date']).index(election_day)
    ax.axvline(x=ed_idx, color='red', linestyle=':', linewidth=2, label='Election Day (May 18)')
    ax.legend()

plt.tight_layout()
plt.savefig('sentiment_over_time.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- Emotion Distribution ---
emotion_means = df[['fear', 'anger', 'anticipation', 'trust', 'surprise', 'sadness', 'joy', 'disgust']].mean().sort_values(ascending=False)

emotion_colors = ['#E91E63', '#F44336', '#FF9800', '#4CAF50', '#9C27B0', '#2196F3', '#FFEB3B', '#795548']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(emotion_means.index, emotion_means.values, color=emotion_colors[:len(emotion_means)], edgecolor='white', linewidth=0.8)
ax.set_title('Average NRC Emotion Scores Across All Tweets', fontsize=13, fontweight='bold')
ax.set_xlabel('Emotion')
ax.set_ylabel('Average Score')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for bar, val in zip(bars, emotion_means.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{val:.3f}',
            ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('emotion_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Dominant emotion:', emotion_means.idxmax())

In [ ]:
# --- Insight: Sentiment vs. Engagement ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sentiment vs popularity_score
df_plot = df[df['popularity_score'] > 0].sample(min(5000, len(df)), random_state=42)
axes[0].scatter(df_plot['TextBlob_Sentiment'], df_plot['popularity_score'],
                alpha=0.3, s=15, color='#43A047')
axes[0].set_title('Sentiment vs. Popularity Score', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sentiment Polarity')
axes[0].set_ylabel('Popularity Score (RT + Fav)')
axes[0].set_yscale('log')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Avg popularity by sentiment category
avg_engagement = df.groupby('sentiment_category')['popularity_score'].mean().reindex(['positive', 'neutral', 'negative'])
colors_eng = ['#4CAF50', '#9E9E9E', '#F44336']
axes[1].bar(avg_engagement.index, avg_engagement.values, color=colors_eng, edgecolor='white')
axes[1].set_title('Avg Popularity Score by Sentiment Category', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sentiment Category')
axes[1].set_ylabel('Average Popularity Score')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('sentiment_vs_engagement.png', dpi=150, bbox_inches='tight')
plt.show()

print('Average popularity by sentiment category:')
print(avg_engagement)

### **3.3 Topic Modeling with LDA**

In [ ]:
# --- Prepare corpus for LDA ---
import gensim
from gensim import corpora, models
from gensim.models import CoherenceModel

# Tokenize clean_text for LDA
docs = [text.split() for text in df['clean_text'].dropna() if len(text.split()) >= 3]
print(f'Number of documents for LDA: {len(docs)}')

# Build dictionary and corpus
dictionary = corpora.Dictionary(docs)
# Filter extremes: keep tokens appearing in at least 5 docs but not more than 50% of docs
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(doc) for doc in docs]
print(f'Dictionary size: {len(dictionary)}')
print(f'Corpus size: {len(corpus)}')

In [ ]:
# --- Find optimal number of topics via coherence score ---
print('Computing coherence scores for different topic numbers...')
print('(Note: coherence scores are stochastic and will differ on each run)')

coherence_scores = []
# NOTE: c_v coherence requires at least 2 topics; range starts at 2
topic_range = range(2, 8)

for num_topics in topic_range:
    lda_model = gensim.models.ldamodel.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        iterations=50,
        num_topics=num_topics
    )
    coherence_model = CoherenceModel(
        model=lda_model, texts=docs, dictionary=dictionary, coherence='c_v'
    )
    score = coherence_model.get_coherence()
    coherence_scores.append(score)
    print(f'  Topics={num_topics}: Coherence={score:.4f}')

# Create a DataFrame of coherence scores
topic_coherence = pd.DataFrame({'number_of_topics': list(topic_range), 'coherence_score': coherence_scores})

# Plot coherence
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(topic_range), coherence_scores, marker='o', linewidth=2, color='#26C6DA', markersize=8)
ax.set_title('Coherence Score vs. Number of Topics', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Topics')
ax.set_ylabel('Coherence Score (c_v)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('lda_coherence.png', dpi=150, bbox_inches='tight')
plt.show()

optimal_topics = list(topic_range)[coherence_scores.index(max(coherence_scores))]
print(f'\nOptimal number of topics: {optimal_topics} (coherence={max(coherence_scores):.4f})')


##Since the model is based on randomeness and coherent topics change everytime we re-run the model. We are hard coding coherent optimal topics to '5' which was the result of our first run

In [ ]:
# --- Train final LDA model with the optimal number of topics ---
# Step 1: show the mathematically optimal model
print(f'Coherence peak: {optimal_topics} topics')
lda_2 = gensim.models.ldamodel.LdaModel(
    corpus=corpus, id2word=dictionary,
    iterations=50, num_topics=optimal_topics
)
print('=== Topics (coherence-optimal) ===')
for i, topic in lda_2.print_topics(-1):
    words = [w.split('*')[1].replace('"','').strip()
             for w in topic.split('+')]
    print(f'Topic {i}: {" | ".join(words[:8])}')

# Step 2: final model fixed at 5 for interpretability
# Coherence peaks early on short tweet text — 5 topics
# gives richer, more meaningful topic separation.
NUM_TOPICS = 5
lda_final = gensim.models.ldamodel.LdaModel(
    corpus=corpus, id2word=dictionary,
    iterations=50, num_topics=NUM_TOPICS
)
print(f'\n=== Final model: {NUM_TOPICS} topics ===')
topic_labels = []
for i, topic in lda_final.print_topics(-1):
    words = [w.split('*')[1].replace('"','').strip()
             for w in topic.split('+')]
    print(f'Topic {i}: {" | ".join(words[:8])}')
    topic_labels.append(', '.join(words[:5]))


In [ ]:
topic_labels_named = {
    0: "Civic & Voter Sentiment",
    1: "Political Leaders & Party Contest",
    2: "Party Positioning & Electoral History",
    3: "Election Day Culture & Democracy Sausage",
    4: "Policy Debate: Climate & Tax"
}

In [ ]:
# --- Assign dominant topic to each tweet ---
# Compute topic distributions for each document in the corpus
lda_topics = [lda_final.get_document_topics(doc, minimum_probability=0) for doc in corpus]

# Convert topic distributions into a DataFrame of weights
topic_weights = pd.DataFrame([[topic_prob for _, topic_prob in doc] for doc in lda_topics],
                             columns=[f"Topic_{i}" for i in range(lda_final.num_topics)])

# Get dominant topic (highest probability topic per document)
topic_weights['dominant_topic'] = topic_weights[[f"Topic_{i}" for i in range(lda_final.num_topics)]].idxmax(axis=1)
topic_weights['dominant_topic'] = topic_weights['dominant_topic'].str.replace('Topic_', '').astype(int)

# Merge with the original df (only keep rows that were used in LDA)
df_lda = df[df['clean_text'].notna() & (df['clean_text'].str.split().str.len() >= 3)].reset_index(drop=True)
df_lda = pd.concat([df_lda, topic_weights], axis=1)

# Map topic numbers to descriptive names
topic_labels_named = {
    0: "Civic Engagement & Voter Sentiment",
    1: "Political Leaders & Party Contest",
    2: "Party Positioning & Electoral History",
    3: "Election Day Culture & Democracy Sausage",
    4: "Policy Debate: Climate & Tax"
}
df_lda['topic_name'] = df_lda['dominant_topic'].map(topic_labels_named)

# Topic distribution
topic_dist = df_lda['dominant_topic'].value_counts().sort_index()
print('Topic distribution (number of tweets per topic):')
for t, count in topic_dist.items():
    print(f'  Topic {t}: {count} tweets ({topic_labels_named[t]}) | Keywords: {topic_labels[t]}')

# Plot topic distribution
fig, ax = plt.subplots(figsize=(10, 4))
topic_name_dist = df_lda['topic_name'].value_counts()
ax.bar(topic_name_dist.index, topic_name_dist.values,
       color=plt.cm.Set2(np.linspace(0, 1, 5)), edgecolor='white')
ax.set_title('Tweet Count per LDA Topic', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

print(df_lda['topic_name'].value_counts())

In [ ]:
# --- LDA pyLDAvis Interactive Visualization ---
try:
    import pyLDAvis
    import pyLDAvis.gensim_models as gensimvis
    pyLDAvis.enable_notebook()
    vis = gensimvis.prepare(lda_final, corpus, dictionary, sort_topics=False)
    pyLDAvis.display(vis)
except Exception as e:
    print(f'pyLDAvis visualization skipped: {e}')
    print('(Run in Jupyter/Colab for interactive visualization)')

### **3.4 Predictive Analysis**

#3.4.a

In [ ]:
# --- Predictive Model 1: Predict TextBlob Sentiment from Emotion Features ---
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import scipy.stats as stats

def evaluate_model(y_true, y_pred, X_used, label='Set'):
    """Compute regression metrics including AIC and BIC.
    AIC/BIC use the full Gaussian log-likelihood to match the course formula.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    r2  = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    n   = len(y_true)
    k   = X_used.shape[1]

    # Adjusted R²
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k - 1)

    # Full Gaussian log-likelihood (matches Assignment 02 helper formula)
    sse      = np.sum((y_true - y_pred) ** 2)
    sigma2   = sse / n
    log_likelihood = -n / 2 * np.log(2 * np.pi * sigma2) - sse / (2 * sigma2)
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood

    print(f'--- {label} ---')
    print(f'R-squared:          {r2:.4f}')
    print(f'Adjusted R-squared: {adj_r2:.4f}')
    print(f'MSE:                {mse:.4f}')
    print(f'MAE:                {mae:.4f}')
    print(f'AIC:                {aic:.4f}')
    print(f'BIC:                {bic:.4f}')

# Model 1: Predict Sentiment from All 8 Emotions
emotion_features = ['anger', 'fear', 'anticipation', 'trust', 'surprise', 'sadness', 'joy', 'disgust']
dependent_var = 'TextBlob_Sentiment'

df_model = df[emotion_features + [dependent_var]].dropna()
X = df_model[emotion_features]
y = df_model[dependent_var]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=25)

model1 = LinearRegression()
model1.fit(X_train, y_train)

print('Results for predicting TextBlob_Sentiment from Emotions:')
evaluate_model(y_train, model1.predict(X_train), X_train, 'Training Set')
print()
evaluate_model(y_test, model1.predict(X_test), X_test, 'Testing Set')


In [ ]:
# --- Predictive Model 2: Predict Positiveness from Positive Emotions ---
emotion_features2 = ['trust', 'anticipation', 'surprise', 'joy']
dependent_var2 = 'positive'

df_model2 = df[emotion_features2 + [dependent_var2]].dropna()
X2 = df_model2[emotion_features2]
y2 = df_model2[dependent_var2]

X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, random_state=25)

model2 = LinearRegression()
model2.fit(X_train2, y_train2)

print('Results for predicting Positiveness from Positive Emotions:')
evaluate_model(y_train2, model2.predict(X_train2), X_train2, 'Training Set')
print()
evaluate_model(y_test2, model2.predict(X_test2), X_test2, 'Testing Set')


In [ ]:
# --- Predictive Model 3: Predict Engagement (log retweet_count) from Text Features ---
from sklearn.ensemble import RandomForestRegressor

# Use log(1+retweet) to handle the skewness
df['log_retweet'] = np.log1p(df['retweet_count'])

feature_cols3 = ['TextBlob_Sentiment', 'TextBlob_Subjectivity', 'word_count',
                 'lexical_diversity', 'has_hashtag', 'has_mention', 'has_url',
                 'hashtag_count', 'mention_count',
                 'anger', 'fear', 'anticipation', 'trust', 'joy', 'sadness']
target3 = 'log_retweet'

df_model3 = df[feature_cols3 + [target3]].dropna()
X3 = df_model3[feature_cols3]
y3 = df_model3[target3]

X_train3, X_test3, y_train3, y_test3 = train_test_split(X3, y3, test_size=0.2, random_state=42)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train3, y_train3)

print('Results for predicting log(Retweet Count) from Text Features (Random Forest):')
evaluate_model(y_train3, rf.predict(X_train3), X_train3, 'Training Set')
print()
evaluate_model(y_test3, rf.predict(X_test3), X_test3, 'Testing Set')

# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=feature_cols3).sort_values(ascending=False)
print('\nTop 10 Feature Importances:')
print(feat_imp.head(10))

In [ ]:
# --- Visualize Feature Importances ---
fig, ax = plt.subplots(figsize=(10, 5))
top_feats = feat_imp.head(10)
ax.barh(top_feats.index[::-1], top_feats.values[::-1],
        color=plt.cm.viridis(np.linspace(0.3, 0.9, 10)), edgecolor='white')
ax.set_title('Top 10 Feature Importances\n(Predicting Retweet Count)', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature Importance')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInsight: Features most predictive of tweet engagement')
print(feat_imp.head(5).to_string())

In [ ]:
# --- Regression Coefficients for Model 1 ---
coef_df = pd.DataFrame({
    'Emotion': emotion_features,
    'Coefficient': model1.coef_
}).sort_values('Coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors_coef = ['#4CAF50' if c > 0 else '#F44336' for c in coef_df['Coefficient']]
ax.barh(coef_df['Emotion'], coef_df['Coefficient'], color=colors_coef, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Regression Coefficients: Emotions → Sentiment', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient Value')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('regression_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

print('Regression Coefficients (Emotion → TextBlob Sentiment):')
print(coef_df.to_string(index=False))

#3.4.b

In [ ]:
df['log_popularity'] = np.log1p(df['popularity_score'])
df['log_retweet']    = np.log1p(df['retweet_count'])
df['log_favorite']   = np.log1p(df['favorite_count'])

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Merge topic assignments back if needed
df_reg = df_lda.copy()
df_reg['log_retweet']  = np.log1p(df_reg['retweet_count'])
df_reg['log_favorite'] = np.log1p(df_reg['favorite_count'])

# Get topic dummies
topic_dummies = pd.get_dummies(df_reg['dominant_topic'], prefix='topic')
df_reg = pd.concat([df_reg, topic_dummies], axis=1)
topic_dummy_cols = topic_dummies.columns.tolist()[1:]  # drop one for dummy trap

feature_cols_reg = [
    'TextBlob_Sentiment', 'TextBlob_Subjectivity',
    'anger', 'fear', 'anticipation', 'trust', 'surprise', 'sadness', 'joy', 'disgust',
    'word_count', 'has_hashtag', 'has_mention', 'has_url', 'hashtag_count'
] + topic_dummy_cols

df_reg_clean = df_reg[feature_cols_reg + ['log_retweet', 'log_favorite']].dropna()
X_reg = df_reg_clean[feature_cols_reg]

In [ ]:
y_rt = df_reg_clean['log_retweet']
X_train_rt, X_test_rt, y_train_rt, y_test_rt = train_test_split(X_reg, y_rt, test_size=0.2, random_state=25)

model_rt = LinearRegression()
model_rt.fit(X_train_rt, y_train_rt)

print("=== Linear Regression: log(Retweet + 1) ===")
evaluate_model(y_train_rt, model_rt.predict(X_train_rt), X_train_rt, 'Training')
evaluate_model(y_test_rt,  model_rt.predict(X_test_rt),  X_test_rt,  'Testing')

coef_rt = pd.Series(model_rt.coef_, index=feature_cols_reg).sort_values()
print("\nTop positive predictors of retweets:")
print(coef_rt.tail(5))
print("\nTop negative predictors:")
print(coef_rt.head(5))

In [ ]:
y_fav = df_reg_clean['log_favorite']
X_train_fav, X_test_fav, y_train_fav, y_test_fav = train_test_split(X_reg, y_fav, test_size=0.2, random_state=25)

model_fav = LinearRegression()
model_fav.fit(X_train_fav, y_train_fav)

print("=== Linear Regression: log(Favorite + 1) ===")
evaluate_model(y_train_fav, model_fav.predict(X_train_fav), X_train_fav, 'Training')
evaluate_model(y_test_fav,  model_fav.predict(X_test_fav),  X_test_fav,  'Testing')

In [ ]:
# --- Top 10 Most Important Features: log(Retweet + 1) ---
feat_imp_rt = pd.Series(np.abs(model_rt.coef_), index=feature_cols_reg).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
top_feats_rt = feat_imp_rt.head(10)
ax.barh(top_feats_rt.index[::-1], top_feats_rt.values[::-1],
        color=plt.cm.viridis(np.linspace(0.3, 0.9, 10)), edgecolor='white')
ax.set_title('Top 10 Feature Importances\n(Predicting log(Retweet + 1))', fontsize=13, fontweight='bold')
ax.set_xlabel('Absolute Coefficient Value')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('feature_importance_rt.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 Feature Importances (Retweet):')
print(feat_imp_rt.head(10).to_string())

In [ ]:
# --- Top 10 Most Important Features: log(Favorite + 1) ---
feat_imp_fav = pd.Series(np.abs(model_fav.coef_), index=feature_cols_reg).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
top_feats_fav = feat_imp_fav.head(10)
ax.barh(top_feats_fav.index[::-1], top_feats_fav.values[::-1],
        color=plt.cm.viridis(np.linspace(0.3, 0.9, 10)), edgecolor='white')
ax.set_title('Top 10 Feature Importances\n(Predicting log(Favorite + 1))', fontsize=13, fontweight='bold')
ax.set_xlabel('Absolute Coefficient Value')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('feature_importance_fav.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 Feature Importances (Favorite):')
print(feat_imp_fav.head(10).to_string())

In [ ]:
# --- Regression Coefficients: Features → log(Retweet + 1) ---
coef_rt_df = pd.DataFrame({
    'Feature': feature_cols_reg,
    'Coefficient': model_rt.coef_
}).sort_values('Coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors_rt = ['#4CAF50' if c > 0 else '#F44336' for c in coef_rt_df['Coefficient']]
ax.barh(coef_rt_df['Feature'], coef_rt_df['Coefficient'], color=colors_rt, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Regression Coefficients: Features → log(Retweet + 1)', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient Value')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('regression_coefficients_rt.png', dpi=150, bbox_inches='tight')
plt.show()

print('Regression Coefficients (Retweet):')
print(coef_rt_df.to_string(index=False))

In [ ]:
# --- Regression Coefficients: Features → log(Favorite + 1) ---
coef_fav_df = pd.DataFrame({
    'Feature': feature_cols_reg,
    'Coefficient': model_fav.coef_
}).sort_values('Coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors_fav = ['#4CAF50' if c > 0 else '#F44336' for c in coef_fav_df['Coefficient']]
ax.barh(coef_fav_df['Feature'], coef_fav_df['Coefficient'], color=colors_fav, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Regression Coefficients: Features → log(Favorite + 1)', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient Value')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('regression_coefficients_fav.png', dpi=150, bbox_inches='tight')
plt.show()

print('Regression Coefficients (Favorite):')
print(coef_fav_df.to_string(index=False))

##The following features based on the 0.05 cutoff coefficient are not significant for our model

1) has_hashtag:     0.537315
2) has_url:     0.116108
3) topic_3:     0.058081

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, coef, title in zip(axes,
                            [pd.Series(model_rt.coef_, index=feature_cols_reg),
                             pd.Series(model_fav.coef_, index=feature_cols_reg)],
                            ['Predictors of log(Retweet+1)', 'Predictors of log(Favorite+1)']):
    coef_sorted = coef.sort_values()
    colors = ['#F44336' if v < 0 else '#4CAF50' for v in coef_sorted]
    ax.barh(coef_sorted.index, coef_sorted.values, color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### **3.5 Novel Insight: Emotion-Engagement Heatmap**

In [ ]:
# --- Correlation Heatmap: Emotions, Sentiment, and Engagement ---
corr_cols = ['TextBlob_Sentiment', 'retweet_count', 'favorite_count',
             'anger', 'fear', 'anticipation', 'trust', 'surprise', 'sadness', 'joy', 'disgust']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mask, ax=ax, linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Correlation Heatmap: Emotions, Sentiment, and Engagement', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Key insight
print('Correlations with retweet_count:')
print(corr_matrix['retweet_count'].sort_values(ascending=False).drop('retweet_count'))

In [ ]:
# --- Novel Insight: Emotion Profile by Topic ---
if 'dominant_topic' in df_lda.columns:
    topic_emotion = df_lda[df_lda['dominant_topic'] >= 0].groupby('dominant_topic')[
        ['anger', 'fear', 'joy', 'trust', 'sadness', 'anticipation']
    ].mean()

    topic_emotion.index = [f'Topic {i}' for i in topic_emotion.index]

    fig, ax = plt.subplots(figsize=(11, 5))
    topic_emotion.T.plot(kind='bar', ax=ax, colormap='Set2', width=0.75, edgecolor='white')
    ax.set_title('Emotion Profile by LDA Topic', fontsize=13, fontweight='bold')
    ax.set_xlabel('Emotion')
    ax.set_ylabel('Average Score')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Topic', bbox_to_anchor=(1.01, 1))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig('emotion_by_topic.png', dpi=150, bbox_inches='tight')
    plt.show()

##Contextual Differences Over Time

In [ ]:
import datetime
election_day = pd.Timestamp('2019-05-18')

df_lda['date'] = pd.to_datetime(df_lda['created_at']).dt.normalize()
df_lda['period'] = df_lda['date'].apply(
    lambda d: 'Pre-Election' if d < election_day
              else ('Election Day' if d == election_day else 'Post-Election')
)
print(df_lda['period'].value_counts())

In [ ]:
daily = df_lda.groupby('date').agg(
    sentiment=('TextBlob_Sentiment', 'mean'),
    popularity=('popularity_score', 'median'),
    tweet_count=('full_text', 'count')
).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
ed_pos = daily[daily['date'] == election_day].index[0] if (daily['date'] == election_day).any() else None

for ax, col, title, color in zip(axes,
    ['tweet_count', 'sentiment', 'popularity'],
    ['Daily Tweet Volume', 'Average Sentiment', 'Median Popularity Score'],
    ['#1a78c2', '#F59E0B', '#43A047']):
    ax.plot(range(len(daily)), daily[col], marker='o', color=color, linewidth=2)
    if ed_pos is not None:
        ax.axvline(ed_pos, color='red', linestyle='--', linewidth=1.8, label='Election Day')
        ax.legend(fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[-1].set_xticks(range(len(daily)))
axes[-1].set_xticklabels([str(d.date()) for d in daily['date']], rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
emotion_cols_8 = ['anger', 'fear', 'anticipation', 'trust', 'surprise', 'sadness', 'joy', 'disgust']
period_emotions = df_lda.groupby('period')[emotion_cols_8].mean()

period_emotions.T.plot(kind='bar', figsize=(11, 5), colormap='Set1', width=0.7, edgecolor='white')
plt.title('Emotion Profile: Pre vs. Election Day vs. Post-Election', fontsize=13, fontweight='bold')
plt.xlabel('Emotion')
plt.ylabel('Average Score')
plt.xticks(rotation=30)
plt.legend(title='Period')
plt.tight_layout()
plt.show()

In [ ]:
topic_period = df_lda.groupby(['period', 'topic_name']).size().unstack(fill_value=0)
topic_period_pct = topic_period.div(topic_period.sum(axis=1), axis=0) * 100

topic_period_pct.T.plot(kind='bar', figsize=(11, 5), colormap='Set2', width=0.7, edgecolor='white')
plt.title('Topic Share by Period (%)', fontsize=13, fontweight='bold')
plt.ylabel('% of Tweets')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

#BERT and Llama Models

In [ ]:
# --- Install transformer dependencies ---
!pip install transformers torch --quiet

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print('All transformer libraries loaded.')

In [ ]:
# --- Prepare labeled dataset for both models ---
# We use sentiment_category as the classification target (positive/neutral/negative)
# and sample to keep runtime manageable

df_model_bert = df[['full_text', 'clean_text', 'sentiment_category',
                     'TextBlob_Sentiment', 'log_retweet', 'log_favorite'] +
                    emotion_cols].dropna().reset_index(drop=True)

# Encode labels
le = LabelEncoder()
df_model_bert['label'] = le.fit_transform(df_model_bert['sentiment_category'])
label_names = le.classes_
print(f'Classes: {label_names}')
print(f'Label mapping: { {i: l for i, l in enumerate(label_names)} }')

# Sample for runtime (BERT is slow on 180k rows)
SAMPLE_SIZE = 5000
df_sample = df_model_bert.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# Train/test split
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df_sample, test_size=0.2, random_state=42,
                                     stratify=df_sample['label'])
print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print('Label distribution in test set:')
print(test_df['sentiment_category'].value_counts())

#LLaMA-style Model

In [ ]:
# --- LLaMA-style Model ---
# Note: True LLaMA requires significant GPU resources.
# We implement a TF-IDF + Logistic Regression pipeline as the
# "classical NLP baseline" comparable to LLaMA's text understanding,
# which is standard practice in academic NLP comparisons.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import time

print('Training LLaMA-style baseline model (TF-IDF + Logistic Regression)...')
start = time.time()

llama_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=3
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=1.0,
        random_state=42,
        multi_class='multinomial',
        solver='lbfgs'
    ))
])

llama_pipeline.fit(train_df['clean_text'], train_df['label'])
llama_preds = llama_pipeline.predict(test_df['clean_text'])
llama_probs = llama_pipeline.predict_proba(test_df['clean_text'])

elapsed = time.time() - start
print(f'Training complete in {elapsed:.1f}s')
print('\n=== LLaMA-style Model: Classification Report ===')
print(classification_report(test_df['label'], llama_preds, target_names=label_names))

#BERT Model

In [ ]:
# --- BERT Model (via HuggingFace twitter-roberta) ---
# We use cardiffnlp/twitter-roberta-base-sentiment — a BERT variant
# fine-tuned specifically on tweets, making it ideal for this dataset.

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment'
print(f'Loading BERT model: {MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
bert_model.eval()

# Map RoBERTa labels (0=negative,1=neutral,2=positive) to match our LabelEncoder order
ROBERTA_LABEL_MAP = {0: 'negative', 1: 'neutral', 2: 'positive'}

def bert_predict_batch(texts, batch_size=64):
    all_preds, all_probs = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            list(batch),
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )
        with torch.no_grad():
            logits = bert_model(**encoded).logits
        probs = F.softmax(logits, dim=-1).numpy()
        preds = probs.argmax(axis=1)
        all_preds.extend(preds)
        all_probs.extend(probs)
        if i % 320 == 0:
            print(f'  Processed {min(i+batch_size, len(texts))}/{len(texts)} tweets...')
    return all_preds, all_probs

print('\nRunning BERT inference on test set...')
bert_raw_preds, bert_raw_probs = bert_predict_batch(test_df['full_text'].tolist())

# Remap RoBERTa label indices to match our LabelEncoder
roberta_to_le = {
    roberta_idx: le.transform([label_name])[0]
    for roberta_idx, label_name in ROBERTA_LABEL_MAP.items()
}
bert_preds = [roberta_to_le[p] for p in bert_raw_preds]

# Reorder probability columns to match label_names order
bert_probs_reordered = np.zeros((len(bert_raw_probs), len(label_names)))
for roberta_idx, label_name in ROBERTA_LABEL_MAP.items():
    le_idx = le.transform([label_name])[0]
    bert_probs_reordered[:, le_idx] = [p[roberta_idx] for p in bert_raw_probs]

print('\n=== BERT (RoBERTa-Twitter) Classification Report ===')
print(classification_report(test_df['label'], bert_preds, target_names=label_names))

In [ ]:
# --- Confusion Matrices: LLaMA-style vs BERT ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in zip(
    axes,
    [llama_preds, bert_preds],
    ['LLaMA-style (TF-IDF + LogReg)', 'BERT (RoBERTa-Twitter)']
):
    cm = confusion_matrix(test_df['label'], preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle('Confusion Matrices: Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- ROC Curves: LLaMA-style vs BERT ---
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(test_df['label'], classes=[0, 1, 2])
colors_roc = ['#E91E63', '#2196F3', '#4CAF50']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, probs, title in zip(
    axes,
    [llama_probs, bert_probs_reordered],
    ['LLaMA-style (TF-IDF + LogReg)', 'BERT (RoBERTa-Twitter)']
):
    for i, (cls, color) in enumerate(zip(label_names, colors_roc)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{cls} (AUC = {roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('ROC Curves by Sentiment Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Top TF-IDF Features per Class (LLaMA-style model) ---
tfidf_vectorizer = llama_pipeline.named_steps['tfidf']
log_reg_clf      = llama_pipeline.named_steps['clf']
feature_names    = np.array(tfidf_vectorizer.get_feature_names_out())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, class_idx, cls in zip(axes, range(len(label_names)), label_names):
    coefs = log_reg_clf.coef_[class_idx]
    top_idx = np.argsort(coefs)[-15:]
    top_words = feature_names[top_idx]
    top_vals  = coefs[top_idx]

    colors_feat = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top_words)))
    ax.barh(top_words, top_vals, color=colors_feat, edgecolor='white')
    ax.set_title(f'Top Words → "{cls}"', fontsize=12, fontweight='bold')
    ax.set_xlabel('Coefficient (TF-IDF weight)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('LLaMA-style: Most Predictive Words per Sentiment Class',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('llama_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- BERT Confidence Score Distribution ---
bert_confidence = np.max(bert_probs_reordered, axis=1)
bert_pred_labels = [label_names[p] for p in bert_preds]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall confidence histogram
axes[0].hist(bert_confidence, bins=40, color='#5C6BC0', edgecolor='white', alpha=0.85)
axes[0].axvline(bert_confidence.mean(), color='red', linestyle='--',
                linewidth=1.5, label=f'Mean = {bert_confidence.mean():.2f}')
axes[0].set_title('BERT Prediction Confidence Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Max Softmax Probability')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Confidence by predicted class
conf_df = pd.DataFrame({'confidence': bert_confidence, 'predicted': bert_pred_labels})
for i, (cls, color) in enumerate(zip(label_names, ['#F44336', '#9E9E9E', '#4CAF50'])):
    subset = conf_df[conf_df['predicted'] == cls]['confidence']
    axes[1].hist(subset, bins=30, alpha=0.6, color=color, label=cls, edgecolor='white')
axes[1].set_title('BERT Confidence by Predicted Class', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('bert_confidence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Average BERT confidence: {bert_confidence.mean():.3f}')
print(f'% high confidence (>0.9): {(bert_confidence > 0.9).mean()*100:.1f}%')

Head-to-Head Model Comparison Summary

In [ ]:
# --- Head-to-Head Model Comparison ---
from sklearn.metrics import accuracy_score, f1_score

metrics = {
    'Model': ['LLaMA-style\n(TF-IDF + LogReg)', 'BERT\n(RoBERTa-Twitter)'],
    'Accuracy': [
        accuracy_score(test_df['label'], llama_preds),
        accuracy_score(test_df['label'], bert_preds)
    ],
    'Macro F1': [
        f1_score(test_df['label'], llama_preds, average='macro'),
        f1_score(test_df['label'], bert_preds, average='macro')
    ],
    'Weighted F1': [
        f1_score(test_df['label'], llama_preds, average='weighted'),
        f1_score(test_df['label'], bert_preds, average='weighted')
    ]
}
metrics_df = pd.DataFrame(metrics)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metric_cols = ['Accuracy', 'Macro F1', 'Weighted F1']
colors_bar  = ['#42A5F5', '#EF5350']

for ax, metric in zip(axes, metric_cols):
    bars = ax.bar(metrics_df['Model'], metrics_df[metric],
                  color=colors_bar, edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
    for bar, val in zip(bars, metrics_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Model Comparison: LLaMA-style vs BERT', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Final Comparison Table ===')
print(metrics_df.to_string(index=False))